# Group 2: Generalization Test Data Prediction

**Team Members:** Aidan Nestor, Andrey Baklykov, Noah Miller, Sanjavan Ghodasara

This notebook implements Section 4 of the project specification:
1. Load test data and apply the saved preprocessing pipeline (Section 3.1)
2. Predict each test company's cluster ID using the saved model (Section 3.3.1)
3. Predict each test company's bankruptcy status using the saved subgroup stacking models (Section 3.3.2)
4. Output `Group2_Generalization.csv` (Table 2 submission file)

In [9]:
import numpy as np
import pandas as pd
import joblib
import os
import warnings
from sklearn.preprocessing import StandardScaler
warnings.filterwarnings('ignore')

PROJECT_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))
DATA_DIR = os.path.join(PROJECT_DIR, 'data')
MODEL_DIR = os.path.join(PROJECT_DIR, 'models')
SUBGROUP_MODEL_DIR = os.path.join(MODEL_DIR, 'subgroup_models')


class PreprocessingPipeline:
    def __init__(self, feature_cols, low_var_cols, corr_drop_cols,
                 low_corr_drop_cols, clip_bounds, random_state=42):
        self.feature_cols = feature_cols
        self.low_var_cols = low_var_cols
        self.corr_drop_cols = corr_drop_cols
        self.low_corr_drop_cols = low_corr_drop_cols
        self.clip_bounds = clip_bounds
        self.scaler = StandardScaler()
        self.clustering_features = None
        self.is_fitted = False
        self.random_state = random_state

    def fit(self, df):
        X = self._transform_raw(df)
        self.clustering_features = X.columns.tolist()
        self.scaler.fit(X)
        self.is_fitted = True
        return self

    def transform(self, df):
        X = self._transform_raw(df)
        X_scaled = pd.DataFrame(
            self.scaler.transform(X),
            columns=self.clustering_features,
            index=X.index
        )
        return X_scaled

    def fit_transform(self, df):
        self.fit(df)
        return self.transform(df)

    def _transform_raw(self, df):
        cols_available = [c for c in self.feature_cols if c in df.columns]
        X = df[cols_available].copy()
        X.columns = X.columns.str.strip()
        for col, (lo, hi) in self.clip_bounds.items():
            if col in X.columns:
                X[col] = X[col].clip(lower=lo, upper=hi)
        X = X.drop(columns=[c for c in self.low_var_cols if c in X.columns])
        X = X.drop(columns=[c for c in self.corr_drop_cols if c in X.columns])
        X = X.drop(columns=[c for c in self.low_corr_drop_cols if c in X.columns])
        return X

print(f'Project directory: {PROJECT_DIR}')

Project directory: /Users/noahmiller/Documents/CS/CS-559/Project



## Step 1 Load Test Data

In [2]:
df_test = pd.read_csv(os.path.join(DATA_DIR, 'test_data.csv'))
df_test.columns = df_test.columns.str.strip()   # test CSV has leading spaces in column names
test_index = df_test['Index'].values.copy()

print(f'Test data shape: {df_test.shape}')
print(f'Index range: {test_index.min()} – {test_index.max()}')
print(f'Columns: {df_test.shape[1]} (Index + {df_test.shape[1] - 1} features)')

Test data shape: (1012, 96)
Index range: 0 – 1011
Columns: 96 (Index + 95 features)



## Step 2 Apply Saved Preprocessing Pipeline (Section 3.1)

The pipeline applies the same transformations used on training data:
feature selection → outlier clipping → drop low-variance / correlated / low-correlation columns → StandardScaler.

In [3]:
pipeline = joblib.load(os.path.join(MODEL_DIR, 'preprocessing_pipeline.joblib'))

X_test = pipeline.transform(df_test)

print(f'Preprocessed test matrix shape: {X_test.shape}')
print(f'Clustering features ({len(pipeline.clustering_features)}): {pipeline.clustering_features[:5]} ...')

Preprocessed test matrix shape: (1012, 50)
Clustering features (50): ['Research and development expense rate', 'Cash flow rate', 'Tax rate (A)', 'Net Value Per Share (A)', 'Persistent EPS in the Last Four Seasons'] ...


## Step 3 Predict Cluster IDs (Section 3.3.1)

In [4]:
cluster_id_model = joblib.load(os.path.join(MODEL_DIR, 'cluster_id_model.joblib'))
cluster_meta = joblib.load(os.path.join(MODEL_DIR, 'cluster_metadata.joblib'))

test_cluster_ids = cluster_id_model.predict(X_test)

print(f'Cluster-ID model type: {type(cluster_id_model).__name__}')
print(f'Constant-prediction clusters: {cluster_meta["constant_clusters"]}')
print(f'Model clusters: {cluster_meta["model_clusters"]}')
print(f'\nTest cluster distribution:')
cluster_counts = pd.Series(test_cluster_ids).value_counts().sort_index()
for cid, cnt in cluster_counts.items():
    label = '(constant → 0)' if cid in cluster_meta['constant_clusters'] else ''
    print(f'  Cluster {cid}: {cnt:4d} companies {label}')

Cluster-ID model type: GradientBoostingClassifier
Constant-prediction clusters: [np.int64(4)]
Model clusters: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(5)]

Test cluster distribution:
  Cluster 0:   72 companies 
  Cluster 1:  144 companies 
  Cluster 2:   68 companies 
  Cluster 3:  332 companies 
  Cluster 4:  228 companies (constant → 0)
  Cluster 5:  168 companies 



## Step 4 Predict Bankruptcy per Subgroup (Section 3.3.2)

For each cluster:
- **Constant clusters** (e.g., C4 with 0 bankrupt in training): predict all 0
- **Model clusters**: load the saved subgroup stacking model, select its features from the *raw* test data, apply any per-model preprocessing (e.g., StandardScaler for C2), get `predict_proba`, and threshold

In [5]:
# Initialize predictions array — default to 0 (non-bankrupt)
y_test_pred = np.zeros(len(df_test), dtype=int)

constant_clusters = cluster_meta['constant_clusters']
model_clusters = cluster_meta['model_clusters']

# Track which clusters have models available
available_models = []
missing_models = []

for c in sorted(set(test_cluster_ids)):
    mask = test_cluster_ids == c
    n_in_cluster = mask.sum()
    
    # Constant clusters: predict all 0
    if c in constant_clusters:
        print(f'Cluster {c}: {n_in_cluster} companies → constant predictor (all 0)')
        continue
    
    # Model clusters: load subgroup model
    model_path = os.path.join(SUBGROUP_MODEL_DIR, f'cluster_{c}_model.joblib')
    if not os.path.exists(model_path):
        missing_models.append(c)
        print(f'Cluster {c}: {n_in_cluster} companies → ⚠ MODEL NOT FOUND — defaulting to 0')
        continue
    
    model_info = joblib.load(model_path)
    available_models.append(c)
    
    stacker = model_info['model']
    features = model_info['features']
    threshold = model_info['threshold']
    scaler = model_info.get('scaler', None)  # C2 has its own scaler; C1, C5 don't
    member = model_info.get('member', 'Unknown')
    
    # Select subgroup features from raw test data
    X_cluster = df_test.loc[mask, features].values
    
    # Apply per-subgroup scaler if present
    if scaler is not None:
        X_cluster = scaler.transform(X_cluster)
    
    # Predict probabilities and threshold
    probas = stacker.predict_proba(X_cluster)[:, 1]
    preds = (probas >= threshold).astype(int)
    y_test_pred[mask] = preds
    
    n_bankrupt = preds.sum()
    print(f'Cluster {c}: {n_in_cluster} companies → {n_bankrupt} predicted bankrupt '
          f'({n_bankrupt/n_in_cluster*100:.1f}%) | thresh={threshold:.2f} | '
          f'N_features={len(features)} | model by {member}')

if missing_models:
    print(f'\n⚠ Missing subgroup models for clusters: {missing_models}')
    print('  These clusters default to predicting all 0 (non-bankrupt).')
    print('  Re-run this notebook once those models are saved.')

Cluster 0: 72 companies → ⚠ MODEL NOT FOUND — defaulting to 0
Cluster 1: 144 companies → 38 predicted bankrupt (26.4%) | thresh=0.49 | N_features=3 | model by Noah Miller
Cluster 2: 68 companies → 36 predicted bankrupt (52.9%) | thresh=0.32 | N_features=6 | model by Sanjavan Ghodasara
Cluster 3: 332 companies → ⚠ MODEL NOT FOUND — defaulting to 0
Cluster 4: 228 companies → constant predictor (all 0)
Cluster 5: 168 companies → 39 predicted bankrupt (23.2%) | thresh=0.45 | N_features=10 | model by Noah Miller

⚠ Missing subgroup models for clusters: [np.int64(0), np.int64(3)]
  These clusters default to predicting all 0 (non-bankrupt).
  Re-run this notebook once those models are saved.


## Step 5 Prediction Summary & Sparsity Check

In [6]:
total_bankrupt = y_test_pred.sum()
sparsity = total_bankrupt / len(y_test_pred)

print(f'Total test companies:      {len(y_test_pred)}')
print(f'Predicted bankrupt:        {total_bankrupt}')
print(f'Predicted non-bankrupt:    {len(y_test_pred) - total_bankrupt}')
print(f'Sparsity (% bankrupt):     {sparsity:.4f} ({sparsity*100:.2f}%)')
print(f'Sparsity constraint (<20%): {"PASS" if sparsity < 0.20 else "FAIL"}')

print(f'\nPer-cluster breakdown:')
for c in sorted(set(test_cluster_ids)):
    mask = test_cluster_ids == c
    n = mask.sum()
    nb = y_test_pred[mask].sum()
    print(f'  Cluster {c}: {n:4d} companies, {nb:3d} bankrupt ({nb/n*100:5.1f}%)')

Total test companies:      1012
Predicted bankrupt:        113
Predicted non-bankrupt:    899
Sparsity (% bankrupt):     0.1117 (11.17%)
Sparsity constraint (<20%): PASS

Per-cluster breakdown:
  Cluster 0:   72 companies,   0 bankrupt (  0.0%)
  Cluster 1:  144 companies,  38 bankrupt ( 26.4%)
  Cluster 2:   68 companies,  36 bankrupt ( 52.9%)
  Cluster 3:  332 companies,   0 bankrupt (  0.0%)
  Cluster 4:  228 companies,   0 bankrupt (  0.0%)
  Cluster 5:  168 companies,  39 bankrupt ( 23.2%)



## Step 6 Generate Submission File (Table 2)

In [7]:
submission = pd.DataFrame({
    'Index': test_index,
    'Bankrupt?': y_test_pred
})

# Sort by Index for clean output
submission = submission.sort_values('Index').reset_index(drop=True)

output_path = os.path.join(PROJECT_DIR, 'Group2_Generalization.csv')
submission.to_csv(output_path, index=False)

print(f'Submission file saved to: {output_path}')
print(f'Shape: {submission.shape}')
print(f'\nFirst 10 rows:')
submission.head(10)

Submission file saved to: /Users/noahmiller/Documents/CS/CS-559/Project/Group2_Generalization.csv
Shape: (1012, 2)

First 10 rows:


,Index,Bankrupt?
0,0,0
1,1,0
2,2,0
3,3,0
4,4,0
5,5,0
6,6,0
7,7,0
8,8,0
9,9,0


In [8]:
# Final validation
assert len(submission) == 1012, f'Expected 1012 rows, got {len(submission)}'
assert set(submission.columns) == {'Index', 'Bankrupt?'}, f'Wrong columns: {submission.columns.tolist()}'
assert submission['Bankrupt?'].isin([0, 1]).all(), 'Bankrupt? column must be 0 or 1'
assert submission['Index'].nunique() == 1012, 'Duplicate Index values found'
assert sparsity < 0.20, f'Sparsity constraint violated: {sparsity:.4f} >= 0.20'

print('All validation checks passed.')
print(f'Submission: {len(submission)} companies, {int(submission["Bankrupt?"].sum())} predicted bankrupt')

All validation checks passed.
Submission: 1012 companies, 113 predicted bankrupt
